# Aula 01: Introdução ao NoSQL & Descasamento de Impedância Objeto-Relacional
**Disciplina:** Bancos de Dados Não-Relacionais (ADS16)
**Professor:** Reginaldo Pereira Fernandes
**IFCE -- Campus Tauá** | Tecnologia em Análise e Desenvolvimento de Sistemas

---
## 🎯 Objetivos Desta Prática:
1. Simulação do modelo relacional normalizado (RDBMS com chaves e JOINs).
2. Simulação do modelo orientado a documentos (NoSQL em formato agregado JSON).
3. Análise visual do *Object-Relational Impedance Mismatch* e comparação de trade-offs.

### 1. Modelo de Domínio Orientado a Objetos (Python Dataclasses)
No código da aplicação, trabalhamos naturalmente com objetos ricos e aninhados:

In [1]:
import json
import sqlite3
import time
from dataclasses import dataclass, field
from typing import List

@dataclass
class ItemPedido:
    produto: str
    quantidade: int
    preco_unitario: float

@dataclass
class Pedido:
    id_pedido: str
    data_emissao: str
    itens: List[ItemPedido] = field(default_factory=list)

    @property
    def total(self) -> float:
        return sum(item.quantidade * item.preco_unitario for item in self.itens)

@dataclass
class Cliente:
    id_cliente: int
    nome: str
    email: str
    cidade: str
    uf: str
    pedidos: List[Pedido] = field(default_factory=list)

print("✅ Classes de domínio definidas com sucesso!")

### 2. Abordagem Relacional (RDBMS Tradicional)
Para salvar os dados acima em um banco relacional normalizado (3FN), precisamos fragmentar a entidade em 3 tabelas interligadas por chaves estrangeiras (, , ).

In [2]:
# Criando banco em memória com SQLite
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE clientes (
    id INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    email TEXT NOT NULL,
    cidade TEXT NOT NULL,
    uf TEXT NOT NULL
);

CREATE TABLE pedidos (
    id TEXT PRIMARY KEY,
    cliente_id INTEGER NOT NULL,
    data_emissao TEXT NOT NULL,
    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);

CREATE TABLE itens_pedido (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pedido_id TEXT NOT NULL,
    produto TEXT NOT NULL,
    quantidade INTEGER NOT NULL,
    preco_unitario REAL NOT NULL,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id)
);
""")

# Inserindo dados
cur.execute("INSERT INTO clientes VALUES (1, 'Maria Silva', 'maria@ifce.edu.br', 'Tauá', 'CE')")
cur.execute("INSERT INTO pedidos VALUES ('PED-001', 1, '2026-09-29')")
cur.execute("INSERT INTO itens_pedido VALUES (NULL, 'PED-001', 'Notebook Dell', 1, 4500.00)")
cur.execute("INSERT INTO itens_pedido VALUES (NULL, 'PED-001', 'Mouse Sem Fio', 2, 85.50)")
conn.commit()
print("✅ Dados relacionais inseridos nas 3 tabelas normalizadas!")

### 3. Recuperação Relacional (JOINs Múltiplos)
Para reconstruir o objeto completo de negócio, o banco relacional precisa fazer múltiplos s:

In [3]:
query = """
SELECT c.nome, c.email, p.id, p.data_emissao, i.produto, i.quantidade, i.preco_unitario
FROM clientes c
JOIN pedidos p ON p.cliente_id = c.id
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE c.id = 1
"""
for row in cur.execute(query).fetchall():
    print("Linha retornada:", row)

### 4. Abordagem Orientada a Documentos (Modelo NoSQL / MongoDB)
No modelo NoSQL de documentos, a entidade inteira é armazenada como um único documento agregado JSON/BSON com os pedidos e itens aninhados (*Embedded Documents*):

In [4]:
cliente_documento = {
    "_id": 1,
    "nome": "Maria Silva",
    "email": "maria@ifce.edu.br",
    "endereco": {"cidade": "Tauá", "uf": "CE"},
    "pedidos": [
        {
            "id_pedido": "PED-001",
            "data_emissao": "2026-09-29",
            "itens": [
                {"produto": "Notebook Dell", "quantidade": 1, "preco_unitario": 4500.00},
                {"produto": "Mouse Sem Fio", "quantidade": 2, "preco_unitario": 85.50}
            ],
            "total": 4671.00
        }
    ]
}

print(json.dumps(cliente_documento, indent=2, ensure_ascii=False))
print("
⚡ Leitura atômica em uma única operação O(1) por chave primária, sem necessidade de JOINs!")